# Login runner & certificate cache checks — M1-10, M1-12, M1-1, M1-2, M1-3

Two different "utility-level" pieces of code that aren't callback services: the M1 CLI test tool's shared
OTP-verification parsing (`tools/m1_test_suite/login_runner.py`), and the ABDM public-certificate cache
(`server/crypto.py`) used by nearly every flow that encrypts something for ABDM. Grouped together as
smaller, standalone checks rather than each getting its own notebook file.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "server").exists():
    REPO_ROOT = Path("__file__").resolve().parents[2] if Path("__file__").exists() else Path.cwd().parents[2]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print("repo root on sys.path:", REPO_ROOT)
assert (REPO_ROOT / "server" / "callbacks").exists(), (
    "Couldn't find server/callbacks/ from here -- open this notebook with the repo root as the "
    "Jupyter working directory, or edit REPO_ROOT above by hand."
)

import harness


repo root on sys.path: C:\Users\hp\Desktop\Aayush\repo


---
## M1-10 / M1-12 — malformed `verify_otp` responses must not crash or misreport success

Both fixed in `tools/m1_test_suite/login_runner.py`'s `verify_login_otp()` — the one shared parsing point
7 of the 8 M1 login variants flow through.

- **M1-10:** a `"tokens"` section that's PRESENT but EMPTY (`{}`) used to be silently treated the same as a
  missing token, while the CLI still printed "Login succeeded." — misreporting a login that produced no
  usable token as a success.
- **M1-12:** an `"accounts"`/`"users"` field that comes back as something other than a list (e.g. a plain
  string) used to be passed straight through to `print_accounts()`/`select_account()`, which iterate it
  expecting dicts — for a string that means iterating individual characters and crashing with an
  `AttributeError` the first time `.get(...)` is called on one.

**Test approach:** `prompt()` (would otherwise block on real keyboard input) and `verify_otp()` (would
otherwise make a real ABDM network call) are both stubbed; `verify_login_otp()` itself is called directly,
unmodified.

**Pass criteria:** the empty-tokens case reports failure, not success; the non-list-accounts case doesn't
crash and is treated as zero accounts; a normal, well-formed response still works exactly as before.

In [2]:
from unittest.mock import patch

import tools.m1_test_suite.login_runner as login_runner

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, {"users": [{"name": "Test Patient"}], "tokens": {}})):
    result_10 = login_runner.verify_login_otp(action="phr/web/login/abha", scope=["abha-login"], txn_id="txn-1")

harness.check("empty-but-present 'tokens' section is reported as a failure, not a false 'Login succeeded'", result_10["x_token"] is None)

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, {"accounts": "not actually a list", "token": "real-token-xyz"})):
    try:
        result_12 = login_runner.verify_login_otp(action="profile/login", scope=["abha-login"], txn_id="txn-2")
        crashed = False
    except Exception:
        crashed = True
        result_12 = None

harness.check("non-list 'accounts' field does not crash the CLI", not crashed)
harness.check("non-list 'accounts' is treated as zero accounts, not passed through raw", result_12 is not None and result_12["accounts"] == [])

with patch.object(login_runner, "prompt", lambda label: "123456"), \
     patch.object(login_runner, "encrypt", lambda raw: f"ENC:{raw}"), \
     patch.object(login_runner, "verify_otp", lambda **kw: harness.FakeResponse(200, {"accounts": [{"name": "Real Patient"}], "token": "real-token-abc"})):
    result_normal = login_runner.verify_login_otp(action="profile/login", scope=["abha-login"], txn_id="txn-3")

harness.check("a normal, well-formed response still reports success with the real token/accounts", result_normal["x_token"] == "real-token-abc" and len(result_normal["accounts"]) == 1)


      Verifying OTP...

[FAIL] OTP verification returned a 200/success response but no usable token was found under either 'token' or 'tokens.token' -- treating this as a failure rather than reporting a successful login with nothing to show for it.
PASS -- empty-but-present 'tokens' section is reported as a failure, not a false 'Login succeeded'
      Verifying OTP...

[FAIL] OTP verification returned an 'accounts'/'users' field that isn't a list (got str: 'not actually a list') -- treating this as no accounts returned rather than crashing on it.

[OK] Login succeeded.
      No accounts were returned.
PASS -- non-list 'accounts' field does not crash the CLI
PASS -- non-list 'accounts' is treated as zero accounts, not passed through raw
      Verifying OTP...

[OK] Login succeeded.
      1 account(s) returned:
        [1] Real Patient  |  ABHA Number: None  |  ABHA Address: None
PASS -- a normal, well-formed response still reports success with the real token/accounts


True

---
## M1-1 — the ABDM public certificate cache self-heals within a bounded TTL

**Real-world scenario:** if ABDM ever rotates the RSA key behind their public certificate endpoint, a
long-running server process that cached the OLD certificate would keep using it forever (no restart, no
local signal that a rotation happened — the failure shows up as a generic ABDM-side decrypt error, not
anything distinguishable here). The fix bounds how long the cache can go stale: after
`_CERTIFICATE_TTL_SECONDS` (24h), the next call forces a fresh download.

**Test approach:** rather than actually waiting 24 hours (or temporarily shrinking the real TTL constant
and restarting the server, which is what the manual runbook version of this test requires), this notebook
directly ages the module's own cached timestamp past the TTL window — exercising the exact same `is_stale`
check `get_public_certificate()` runs, without waiting or touching the real constant.

**Pass criteria:** the first call downloads; an immediate second call reuses the cache (no re-download);
a call made after the TTL window has elapsed triggers a fresh download.

In [5]:
import time

import server.crypto as crypto

crypto.clear_certificate_cache()

download_calls = []
def fake_download():
    download_calls.append(time.monotonic())
    return object()

with patch.object(crypto, "download_public_certificate", fake_download):
    crypto.get_public_certificate()
    harness.check("first call downloads (cache was empty)", len(download_calls) == 1)

    crypto.get_public_certificate()
    harness.check("second call right after reuses the cache (no re-download)", len(download_calls) == 1)

    # Age the cached timestamp past the TTL window without actually waiting.
    crypto._cached_at = time.monotonic() - crypto._CERTIFICATE_TTL_SECONDS - 1

    crypto.get_public_certificate()
    harness.check("a call after the TTL window elapses triggers a fresh download (self-heals without a restart)", len(download_calls) == 2)

crypto.clear_certificate_cache()  # leave the module in a clean state for anything run after this


2026-08-14 21:09:30  -> ABDM public certificate downloaded
2026-08-14 21:09:30  -> ABDM public certificate refreshed (TTL expired)


PASS -- first call downloads (cache was empty)
PASS -- second call right after reuses the cache (no re-download)
PASS -- a call after the TTL window elapses triggers a fresh download (self-heals without a restart)


---
## M1-2 — Concurrent logins must not download the ABDM public certificate twice

**Real code under test:** `server/crypto.py` — `get_public_certificate()`.

**Real-world scenario:** two patients complete ABHA login at (almost) the exact same moment on a busy
morning. Both requests hit the certificate cache while it's empty (server just started) or just went stale.
Before this fix, both could see "no cached certificate" and both call `download_public_certificate()` --
wasteful at best, and at worst one caller's own `return _cached_certificate` could read back a DIFFERENT
certificate object than the one it itself just downloaded (whichever write landed last wins), which is a
genuinely confusing bug to chase down later.

**Fix:** a `threading.Lock` around the whole "check staleness, maybe download, write cache" sequence
serializes concurrent callers within this one process -- same pattern as `json_file_store.py`'s per-file
lock (M2-1) and M1-3's token-cache lock below.

**Pass criteria:** 10 concurrent callers racing an artificially slow download → exactly 1 real download,
and every caller gets back the identical certificate object.

In [3]:
import threading
import time
from unittest.mock import patch

import server.crypto as crypto

crypto.clear_certificate_cache()

download_calls = []
class FakeCert:
    def __init__(self, n):
        self.n = n

def slow_download():
    time.sleep(0.1)
    download_calls.append(time.monotonic())
    return FakeCert(len(download_calls))

with patch.object(crypto, "download_public_certificate", slow_download):
    results = []
    def worker():
        results.append(crypto.get_public_certificate())
    threads = [threading.Thread(target=worker) for _ in range(10)]
    [t.start() for t in threads]
    [t.join() for t in threads]

harness.check("10 concurrent callers -> exactly 1 real download (lock serialized them)", len(download_calls) == 1)
harness.check("every caller got back the SAME certificate object (no mismatched-write race)", len(set(id(r) for r in results)) == 1)

crypto.clear_certificate_cache()


2026-08-14 21:08:51  -> ABDM public certificate downloaded


PASS -- 10 concurrent callers -> exactly 1 real download (lock serialized them)
PASS -- every caller got back the SAME certificate object (no mismatched-write race)


---
## M1-3 — Concurrent logins must not scramble the cached ABDM gateway token

**Real code under test:** `server/utils.py` — `get_gateway_token()`.

**Real-world scenario:** same busy-morning setup as M1-2, but for the shared ABDM gateway access token
instead of the certificate. Before this fix, `_token_cache["access_token"]` and
`_token_cache["expires_at"]` were written as two separate statements with no lock -- two concurrent
refreshes could interleave their writes and leave the cache holding one refresh's token paired with a
*different* refresh's expiry, a genuinely mismatched pair that doesn't correspond to either real token
response ABDM actually issued.

**Fix:** same `threading.Lock` pattern as M1-2, wrapped around the whole check-and-maybe-refresh sequence.

**Pass criteria:** 10 concurrent callers racing an artificially slow token refresh → exactly 1 real refresh
call, every caller gets back the identical token string, and the cached expiry is set (the pair landed
together, not split across two different refreshes).

In [4]:
import threading
import time
from unittest.mock import patch

import server.utils as utils

utils._token_cache["access_token"] = None
utils._token_cache["expires_at"] = None

class FakeTokenResponse:
    def __init__(self, body):
        self._body = body
    def json(self):
        return self._body
    def raise_for_status(self):
        pass

gen_calls = []
def slow_generate_gateway_token():
    time.sleep(0.1)
    idx = len(gen_calls)
    gen_calls.append(idx)
    return FakeTokenResponse({"accessToken": f"TOKEN_{idx}", "expiresIn": 1800})

with patch("server.auth.generate_gateway_token", slow_generate_gateway_token):
    results = []
    def worker():
        results.append(utils.get_gateway_token())
    threads = [threading.Thread(target=worker) for _ in range(10)]
    [t.start() for t in threads]
    [t.join() for t in threads]

harness.check("10 concurrent callers -> exactly 1 real token refresh (lock serialized them)", len(gen_calls) == 1)
harness.check("every caller got back the SAME token string (no mismatched token/expiry write race)", len(set(results)) == 1)
harness.check("cached expires_at is set (paired write completed atomically under the lock)", utils._token_cache["expires_at"] is not None)

utils._token_cache["access_token"] = None
utils._token_cache["expires_at"] = None


2026-08-14 21:09:16  -> Gateway token refreshed


PASS -- 10 concurrent callers -> exactly 1 real token refresh (lock serialized them)
PASS -- every caller got back the SAME token string (no mismatched token/expiry write race)
PASS -- cached expires_at is set (paired write completed atomically under the lock)
